**0. CẤU HÌNH & THƯ VIỆN DÙNG CHUNG**

Notebook này tái lập quy trình (tiền xử lý → FastDTW → 3 đặc trưng) và đã sửa các điểm sau so với bản cũ:
- Không còn chọn ngưỡng/mô hình trên tập test.
- Bỏ cặp video giả (giáo viên so với chính nó), chỉ dùng cặp thật.
- Mỗi khung hình học viên chỉ ghép với **đúng 1** khung hình giáo viên sau DTW (bản cũ có khung bị nhân đôi số dòng, điểm có thể vượt 13).
- Đặt seed, in thông tin phần cứng.

In [ ]:
import os, time, hashlib, platform, warnings
import numpy as np
import pandas as pd
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATA_DIR = '.'                      # thư mục chứa student_dance_*.csv / teacher_dance_*.csv
VIS_THRESHOLD = 0.5                 # Visibility < 0.5 -> coi là mất dấu (NaN)
MOVEMENT_STD_THRESHOLD = 0.015      # lọc video học viên đứng yên

JOINTS = ['nose',
          'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist',
          'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle']
N_JOINTS = len(JOINTS)              # 13
FEATURES = ['Joint_Error', 'Joint_Cosine', 'Velocity_Error']
LABEL = 'Ground_Truth_Label'

# 13 cặp video thật (bỏ cặp id=13 là giáo viên so với chính nó; id=14 không dùng)
PAIR_IDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15]
TEST_VIDEO_IDS = [5, 9]            # 2 cặp để test, 11 cặp còn lại để train

# True: loại khỏi train các video có video giáo viên trùng với video test (xem cảnh báo ở phần chia dữ liệu)
EXCLUDE_TRAIN_SHARING_TEACHER_WITH_TEST = False

CATBOOST_PARAMS = dict(iterations=150, depth=6, learning_rate=0.08, random_seed=SEED, verbose=0)

print('Phần cứng:', platform.processor() or platform.machine(), '| CPU logic:', os.cpu_count(),
      '|', platform.system(), platform.release(), '| Python', platform.python_version())

**1. TIỀN XỬ LÝ (VISIBILITY → NỘI SUY → BỎ TRỤC Z → GỐC HÔNG → CHIA THEO THÂN NGƯỜI)**

In [ ]:
def is_dancing(df_student, threshold=MOVEMENT_STD_THRESHOLD):
    """Bộ lọc video đứng yên: độ lệch chuẩn trung bình của cổ tay và cổ chân."""
    key_joints = ['left_wrist', 'right_wrist', 'left_ankle', 'right_ankle']
    d = df_student[df_student['Joint_Name'].isin(key_joints)]
    movement_std = d.groupby('Joint_Name')[['X', 'Y']].std().mean().mean()
    return movement_std >= threshold, movement_std


def preprocess_data(df):
    """Trả về dict: frames, X, Y (đã chuẩn hóa, shape [n_frames, 13]) và label (nếu có)."""
    df = df.sort_values(['Frame', 'Joint_Name']).reset_index(drop=True).copy()

    # Mất dấu -> NaN -> nội suy tuyến tính theo từng khớp (chỉ dùng X, Y; bỏ trục Z)
    df.loc[df['Visibility'] < VIS_THRESHOLD, ['X', 'Y']] = np.nan
    df[['X', 'Y']] = df.groupby('Joint_Name')[['X', 'Y']].transform(
        lambda s: s.interpolate(method='linear', limit_direction='both'))
    df[['X', 'Y']] = df[['X', 'Y']].fillna(0)

    X = df.pivot(index='Frame', columns='Joint_Name', values='X')[JOINTS]
    Y = df.pivot(index='Frame', columns='Joint_Name', values='Y')[JOINTS]

    root_x = (X['left_hip'] + X['right_hip']) / 2
    root_y = (Y['left_hip'] + Y['right_hip']) / 2
    sh_x = (X['left_shoulder'] + X['right_shoulder']) / 2
    sh_y = (Y['left_shoulder'] + Y['right_shoulder']) / 2
    torso = np.maximum(np.hypot(sh_x - root_x, sh_y - root_y), 1e-5)

    out = {
        'frames': X.index.to_numpy(),
        'X': X.sub(root_x, axis=0).div(torso, axis=0).to_numpy(),
        'Y': Y.sub(root_y, axis=0).div(torso, axis=0).to_numpy(),
    }
    if 'Label' in df.columns:
        out['label'] = df.pivot(index='Frame', columns='Joint_Name', values='Label')[JOINTS].to_numpy()
    return out

**2. FASTDTW + TRÍCH XUẤT 3 ĐẶC TRƯNG**

Sau DTW, mỗi khung hình học viên được ghép với đúng **một** khung hình giáo viên (khung có khoảng cách nhỏ nhất trong các khung được ghép). Nhờ vậy mỗi khung luôn có đúng 13 dòng và điểm $S_{frame}\in[0,13]$.
Vận tốc được tính trên chuỗi **đã gióng hàng** (sai phân bậc nhất giữa hai khung liên tiếp của chuỗi đã gióng).

In [ ]:
def align_and_extract(seq_s, seq_t):
    vs = np.hstack([seq_s['X'], seq_s['Y']])
    vt = np.hstack([seq_t['X'], seq_t['Y']])
    _, path = fastdtw(vs, vt, dist=euclidean)

    best = {}                                   # khung học viên i -> (khoảng cách, khung giáo viên j)
    for i, j in path:
        d = np.linalg.norm(vs[i] - vt[j])
        if i not in best or d < best[i][0]:
            best[i] = (d, j)
    t_idx = np.array([best[i][1] for i in range(len(vs))])

    Xs, Ys = seq_s['X'], seq_s['Y']
    Xt, Yt = seq_t['X'][t_idx], seq_t['Y'][t_idx]

    def vel(a):                                  # sai phân bậc nhất, khung đầu = 0
        return np.diff(a, axis=0, prepend=a[:1])

    joint_error = np.hypot(Xs - Xt, Ys - Yt)
    joint_cosine = (Xs * Xt + Ys * Yt) / (np.hypot(Xs, Ys) * np.hypot(Xt, Yt) + 1e-9)
    velocity_error = np.hypot(vel(Xs) - vel(Xt), vel(Ys) - vel(Yt))

    n = len(vs)
    return pd.DataFrame({
        'Frame_s': np.repeat(seq_s['frames'], N_JOINTS),
        'Joint_Name': np.tile(JOINTS, n),
        'Joint_Error': joint_error.ravel(),
        'Joint_Cosine': joint_cosine.ravel(),
        'Velocity_Error': velocity_error.ravel(),
        LABEL: seq_s['label'].ravel().astype(int),
    })


def file_md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

**3. XÂY DỰNG TẬP DỮ LIỆU TỪ CÁC CẶP VIDEO**

In [ ]:
parts, teacher_group = [], {}
print('=== TRÍCH XUẤT ĐẶC TRƯNG ===')
for vid in PAIR_IDS:
    s_path = os.path.join(DATA_DIR, f'student_dance_{vid}.csv')
    t_path = os.path.join(DATA_DIR, f'teacher_dance_{vid}.csv')
    df_s, df_t = pd.read_csv(s_path), pd.read_csv(t_path)

    moving, std = is_dancing(df_s)
    if not moving:
        print(f'[BỘ LỌC] Bỏ video {vid}: đứng yên (Std={std:.4f})')
        continue

    feats = align_and_extract(preprocess_data(df_s), preprocess_data(df_t))
    feats['Video_ID'] = vid
    parts.append(feats)
    teacher_group[vid] = file_md5(t_path)       # video giáo viên giống hệt nhau -> cùng nhóm
    print(f'[OK] Video {vid}: {feats["Frame_s"].nunique()} khung hình')

master_dataset = pd.concat(parts, ignore_index=True)

assert (master_dataset.groupby(['Video_ID', 'Frame_s']).size() == N_JOINTS).all(), 'mỗi khung phải có đúng 13 dòng'

all_ids = sorted(int(v) for v in master_dataset['Video_ID'].unique())
train_ids = [v for v in all_ids if v not in TEST_VIDEO_IDS]
test_ids = [v for v in all_ids if v in TEST_VIDEO_IDS]

test_teachers = {teacher_group[v] for v in test_ids}
overlap = [v for v in train_ids if teacher_group[v] in test_teachers]
if overlap:
    print(f'\n[CẢNH BÁO] Video train {overlap} dùng cùng video giáo viên với video test {test_ids}.')
    if EXCLUDE_TRAIN_SHARING_TEACHER_WITH_TEST:
        train_ids = [v for v in train_ids if v not in overlap]
        print('  -> đã loại khỏi tập train.')
    else:
        print('  -> vẫn giữ trong train (đặt EXCLUDE_TRAIN_SHARING_TEACHER_WITH_TEST=True để loại).')

train_data = master_dataset[master_dataset['Video_ID'].isin(train_ids)].reset_index(drop=True)
test_data = master_dataset[master_dataset['Video_ID'].isin(test_ids)].reset_index(drop=True)

n_tr = train_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]
n_te = test_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]
print(f'\nTRAIN: {len(train_ids)} cặp {train_ids} | {n_tr:,} khung hình')
print(f'TEST : {len(test_ids)} cặp {test_ids} | {n_te:,} khung hình')
print('Tỉ lệ nhãn khớp (1=đúng) train/test:',
      round(train_data[LABEL].mean(), 3), '/', round(test_data[LABEL].mean(), 3))

**4. SO SÁNH ĐA MÔ HÌNH (CẤP KHỚP) — KHÔNG CHỌN MÔ HÌNH TRÊN TẬP TEST**

- Tất cả mô hình huấn luyện trên TRAIN và đánh giá trên TEST đúng một lần; kết quả chỉ để mô tả, **không** dùng để chọn mô hình hay siêu tham số.
- CatBoost dùng đúng `CATBOOST_PARAMS` như ở `processing_11`, nên số liệu hàng CatBoost nhất quán với kết quả cấp khung hình.
- Thời gian huấn luyện chỉ tính `fit()` (không gồm dự đoán và tính chỉ số).
- Vì chỉ có 1 lần chia và test chỉ có 2 video, chênh lệch nhỏ giữa các mô hình cây không có ý nghĩa thống kê; phần 4.1 lặp với nhiều seed để thấy độ dao động.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


def build_models(seed=SEED):
    base = {
        'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=seed),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=seed),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=seed),
        'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=seed, eval_metric='logloss'),
        'LightGBM': LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=seed, verbose=-1),
        'CatBoost': CatBoostClassifier(**{**CATBOOST_PARAMS, 'random_seed': seed}),
    }
    # Voting: bỏ KNN vì khác bản chất với nhóm cây
    base['Ensemble (Soft Voting)'] = VotingClassifier(
        estimators=[(n, m) for n, m in build_models_for_voting(seed).items()], voting='soft')
    return base


def build_models_for_voting(seed):
    return {
        'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=seed),
        'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=seed),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=seed),
        'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=seed, eval_metric='logloss'),
        'LightGBM': LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=seed, verbose=-1),
        'CatBoost': CatBoostClassifier(**{**CATBOOST_PARAMS, 'random_seed': seed}),
    }


X_train, y_train = train_data[FEATURES], train_data[LABEL]
X_test, y_test = test_data[FEATURES], test_data[LABEL]
print(f'Joint-level: train {len(X_train):,} mẫu | test {len(X_test):,} mẫu | '
      f'baseline lớp đa số trên test = {100 * max(y_test.mean(), 1 - y_test.mean()):.2f}%')

In [ ]:
# Khung hình mẫu (13 khớp) để đo độ trễ: khung đầu tiên của tập test
sample_frame = test_data.iloc[:N_JOINTS][FEATURES]
N_ITER = 1000

results, trained = [], {}
for name, model in build_models().items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)                       # chỉ tính thời gian fit
    train_time = time.perf_counter() - t0
    trained[name] = model

    pred = np.asarray(model.predict(X_test)).ravel().astype(int)

    for _ in range(10):                               # warm-up
        model.predict(sample_frame)
    t0 = time.perf_counter()
    for _ in range(N_ITER):
        model.predict(sample_frame)
    latency_ms = (time.perf_counter() - t0) / N_ITER * 1000

    results.append({
        'Mô hình': name,
        'Accuracy (%)': 100 * accuracy_score(y_test, pred),
        'Macro-F1': f1_score(y_test, pred, average='macro'),
        'Train (s)': train_time,
        'Độ trễ/khung (ms)': latency_ms,
        'FPS': 1000 / latency_ms,
    })

table1 = pd.DataFrame(results)
print(table1.round({'Accuracy (%)': 2, 'Macro-F1': 4, 'Train (s)': 3, 'Độ trễ/khung (ms)': 4, 'FPS': 0}).to_string(index=False))
print('\n* Độ trễ chỉ gồm bước phân loại 13 khớp (chưa gồm MediaPipe, chuẩn hóa, FastDTW).')
print('* Chuẩn thời gian thực tham chiếu: >= 30 FPS (<= 33.3 ms/khung).')

**4.1 Độ dao động theo seed** (lặp 5 seed cho các mô hình có tính ngẫu nhiên; KNN không phụ thuộc seed). Dùng để xem chênh lệch giữa các mô hình cây có lớn hơn nhiễu ngẫu nhiên hay không.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
rows = []
for name in ['Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost', 'LightGBM', 'CatBoost']:
    accs, f1s = [], []
    for sd in SEEDS:
        m = build_models(sd)[name]
        m.fit(X_train, y_train)
        p = np.asarray(m.predict(X_test)).ravel().astype(int)
        accs.append(100 * accuracy_score(y_test, p))
        f1s.append(f1_score(y_test, p, average='macro'))
    rows.append({'Mô hình': name,
                 'Accuracy (%)': f'{np.mean(accs):.2f} ± {np.std(accs):.2f}',
                 'Macro-F1': f'{np.mean(f1s):.4f} ± {np.std(f1s):.4f}'})
print(pd.DataFrame(rows).to_string(index=False))